# NYC Mobility - Source Ingestion

## What ingestion means

Ingestion is simply how we bring source data into our storage before transforming it. We preserve the source first so we can always trace what arrived.


## Green Taxi

The monthly Parquet files were placed in the Green Taxi source folder in R2. No automated download code was included in the original notebooks, so we do not invent one here.

Files already referenced by the project:

- `green_tripdata_2026-03.parquet`
- `green_tripdata_2026-04.parquet`
- `green_tripdata_2026-05.parquet`


## Taxi Zones

The exact file `taxi_zone_lookup.csv` was manually placed in:

`/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/taxi_zones/`

The original work does not include automated acquisition code, so we document the real landing step and do not invent a download process.


## Weather API ingestion

We used the Open-Meteo Archive API because the assignment period is historical: March 1 through May 31, 2026. The response is saved as raw JSON without flattening because cleaning and reshaping belong after ingestion.


In [0]:
%python
import requests
from pathlib import Path

api_url = (
    "https://archive-api.open-meteo.com/v1/archive"
    "?latitude=40.7128"
    "&longitude=-74.006"
    "&start_date=2026-03-01"
    "&end_date=2026-05-31"
    "&hourly=temperature_2m,precipitation,rain,snowfall,weather_code,wind_speed_10m"
    "&timezone=America%2FNew_York"
)

output_path = (
    "/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/"
    "groups/week-08/group-b/source/weather/"
    "open_meteo_2026-03-01_2026-05-31.json"
)

response = requests.get(api_url, timeout=60)
response.raise_for_status()

Path(output_path).parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "wb") as file:
    file.write(response.content)

print("HTTP status:", response.status_code)
print("Saved to:", output_path)
print("Bytes:", len(response.content))

HTTP status: 200
Saved to: /Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/weather/open_meteo_2026-03-01_2026-05-31.json
Bytes: 100500


The original run returned **HTTP 200** and saved the exact file `open_meteo_2026-03-01_2026-05-31.json` in the Weather source folder. We keep the API response raw so we always retain what the source originally returned. The filename will also identify the Bronze batch.


## Traffic Advisory web scraping

This is the optional bonus source. Each scraper run creates a new UTC timestamp batch ID, saves the HTML exactly as received, and saves a separate metadata JSON file.

The verified saved batch is `20260914T040846Z`. We use these exact files for the Bronze load:

- `nyc_dot_weekend_traffic_20260914T040846Z.html`
- `nyc_dot_weekend_traffic_20260914T040846Z.metadata.json`

For the Bronze idempotency test, we rerun the load against these same saved files. We do not rerun the scraper because that would create a different batch.


In [0]:
import requests
from datetime import datetime, timezone
from pathlib import Path
import json

source_url = "https://www.nyc.gov/html/dot/html/motorist/wkndtraf.shtml"

base_path = (
    "/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/"
    "groups/week-08/group-b/source/traffic_advisory"
)

# unique scrape timestamp
scraped_at = datetime.now(timezone.utc)
batch_id = scraped_at.strftime("%Y%m%dT%H%M%SZ")

html_path = f"{base_path}/nyc_dot_weekend_traffic_{batch_id}.html"
metadata_path = f"{base_path}/nyc_dot_weekend_traffic_{batch_id}.metadata.json"

Path(base_path).mkdir(parents=True, exist_ok=True)

response = requests.get(
    source_url,
    timeout=60,
    headers={
        "User-Agent": "FTW-B12-Data-Engineering-Course-Project"
    }
)

response.raise_for_status()

# save the raw HTML exactly as received
with open(html_path, "wb") as file:
    file.write(response.content)

# save provenance separately
metadata = {
    "source_system": "nyc_dot",
    "source_url": source_url,
    "batch_id": batch_id,
    "ingested_at_utc": scraped_at.isoformat(),
    "http_status": response.status_code,
    "content_type": response.headers.get("Content-Type"),
    "bytes_received": len(response.content)
}

with open(metadata_path, "w") as file:
    json.dump(metadata, file, indent=2)

print("HTTP status:", response.status_code)
print("HTML saved to:", html_path)
print("Metadata saved to:", metadata_path)
print("Bytes received:", len(response.content))

HTTP status: 200
HTML saved to: /Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/traffic_advisory/nyc_dot_weekend_traffic_20260914T040846Z.html
Metadata saved to: /Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/traffic_advisory/nyc_dot_weekend_traffic_20260914T040846Z.metadata.json
Bytes received: 53391


The original run returned **HTTP 200**, saved 53,391 bytes of raw HTML, and wrote a 295-byte metadata file. BeautifulSoup inspection belongs in the source-inspection notebook. Event and road parsing does not belong in ingestion or Bronze.


## Ingestion Summary

Green Taxi Parquet files and the exact Taxi Zone CSV were manually placed in their R2 source folders. Weather arrived through the Archive API, and the optional Traffic Advisory arrived through an HTTP scrape. We preserved the exact source filenames and raw responses; no source was cleaned or expanded here.
